In [7]:
!pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp310-cp310-linux_x86_64.whl size=2357274 sha256=c855fcc4bc05d8996bfbaf94abf22b00e68ce6be29c992307ccae58fc98c72fe
  Stored in directory: /root/.cache/pip/wheels/4b/3f/df/6acbf0a40397d9bf3ff97f582cc22fb9ce66adde75bc71fd54
Successfully built scikit-surprise


In [55]:
import pickle
import re
import nltk
import pandas as pd

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

nltk.download("stopwords")
nltk.download("punkt")


def fix_merged_columns(row):
    merged_value = row['Book-Title']
    pattern = r'^(.+?)\\";(.+)"$'
    match = re.match(pattern, merged_value)
    if match:
        author, title = match.groups()
        return author.strip(), title.strip()
    else:
        return None, None


def books_preprocessing(books: pd.DataFrame) -> pd.DataFrame:
    new_book = books[books["Year-Of-Publication"].map(str).str.match("[^0-9]")]
    books['Year-Of-Publication'] = books['Year-Of-Publication'].astype(str)
    mask = ~books["Year-Of-Publication"].str.contains(r'\D')
    books_filtered = books[mask]
    new_book['Book-Title'].unique()
    fixed_data = new_book.apply(lambda x: fix_merged_columns(x), axis=1)
    new_book['Publisher'] = new_book['Year-Of-Publication']
    new_book['Year-Of-Publication'] = new_book['Book-Author']
    new_book[['Book-Title', 'Book-Author']] = fixed_data.tolist()
    books = pd.concat([books_filtered, new_book])
    books['Year-Of-Publication'] = pd.to_numeric(books['Year-Of-Publication'], downcast='integer')
    books = books.drop('Image-URL-S', axis=1)
    books = books.drop('Image-URL-M', axis=1)
    books = books.drop('Image-URL-L', axis=1)
    return books
    pass


def ratings_preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    ratings = df.rename(columns={'Book-Rating': 'Rating'})
    ratings['Rating'] = ratings['Rating'].astype(float)
    ratings = ratings.query("Rating != 0.0")
    min_ratings = 2
    book_counts = ratings.groupby('ISBN')['User-ID'].nunique()
    user_counts = ratings.groupby('User-ID')['ISBN'].nunique()
    good_books = book_counts[book_counts >= min_ratings].index
    good_users = user_counts[user_counts >= min_ratings].index
    filtered1_ratings = ratings[(ratings['ISBN'].isin(good_books)) & (ratings['User-ID'].isin(good_users))]
    return filtered1_ratings
    pass


def title_preprocessing(text: str) -> str:
    tokens = nltk.word_tokenize(text.lower())
    filtered_tokens = [token for token in tokens if token not in stopwords and token.isalpha()]
    return ' '.join(filtered_tokens)
    pass


def modeling(books: pd.DataFrame, ratings: pd.DataFrame) -> None:
    categorical_features = ['Book-Author', 'Publisher']
    numerical_features = ['Year-Of-Publication']
    for feature in categorical_features:
        books[feature] = books[feature].astype('category').cat.codes
    avg_ratings = ratings[['ISBN', 'Rating']].drop_duplicates()
    final_df = books.merge(avg_ratings, on='ISBN', how='inner')
    X = final_df[numerical_features + categorical_features]
    y = final_df['Rating']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=8)
    vectorizer = TfidfVectorizer(max_features=3)
    titles = final_df['Book-Title']  # .apply(title_preprocessing)
    X_titles_train = vectorizer.fit_transform(titles.iloc[X_train.index])
    X_titles_test = vectorizer.transform(titles.iloc[X_test.index])
    X_train = pd.concat([X_train.reset_index(drop=True), pd.DataFrame(X_titles_train.toarray())], axis=1)
    X_test = pd.concat([X_test.reset_index(drop=True), pd.DataFrame(X_titles_test.toarray())], axis=1)
    X_train.columns = X_train.columns.astype(str)
    X_test.columns = X_test.columns.astype(str)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    linreg = LogisticRegression()
    linreg.fit(X_train_scaled, y_train)
    y_pred = linreg.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    with open("linreg.pkl", "wb") as file:
        pickle.dump(linreg, file)

books1 = pd.read_csv("Books.csv")
ratings1 = pd.read_csv("Ratings.csv")
filtered_ratings1 = ratings_preprocessing(ratings1)
filtered_books1 = books_preprocessing(books1)
modeling(filtered_books1, filtered_ratings1)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Mean Absolute Error (MAE): 1.4328


In [18]:
import pandas as pd
import pickle
from surprise import accuracy
from surprise import SVD
from surprise import Dataset, Reader
from surprise.model_selection import train_test_split


def ratings_preprocessing(df: pd.DataFrame) -> pd.DataFrame:
    ratings = df.rename(columns={'Book-Rating': 'Rating'})
    ratings = ratings.query("Rating != 0")
    min_ratings = 2
    book_counts = ratings.groupby('ISBN')['User-ID'].nunique()
    user_counts = ratings.groupby('User-ID')['ISBN'].nunique()
    good_books = book_counts[book_counts >= min_ratings].index
    good_users = user_counts[user_counts >= min_ratings].index
    filtered1_ratings = ratings[(ratings['ISBN'].isin(good_books)) & (ratings['User-ID'].isin(good_users))]
    return filtered1_ratings


def modeling(ratings: pd.DataFrame) -> None:
    reader = Reader(rating_scale=(1, 10))
    data = Dataset.load_from_df(ratings[['User-ID', 'ISBN', 'Rating']], reader)
    train_set, test_set = train_test_split(data, test_size=0.3)
    svd = SVD(n_factors=128, n_epochs=15,  verbose=True)
    svd.fit(train_set)

    predictions = svd.test(test_set)
    mae = accuracy.mae(predictions, verbose=False)
    print(f"Mean Absolute Error (MAE): {mae:.4f}")

    with open("svd.pkl", "wb") as file:
        pickle.dump(svd, file)


test = pd.read_csv("Ratings.csv")
test_1 = ratings_preprocessing(test)
print(test_1)
modeling(test_1)

         User-ID        ISBN  Rating
16        276747  0060517794       9
19        276747  0671537458       9
20        276747  0679776818       8
33        276762  0380711524       5
44        276762  3453092007       8
...          ...         ...     ...
1149745   276688  0892966548      10
1149746   276688  1551669315       6
1149761   276704  0345386108       6
1149771   276704  0743211383       7
1149775   276704  1563526298       9

[270604 rows x 3 columns]
Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Mean Absolute Error (MAE): 1.2608


In [42]:
import pandas
import pickle
import warnings
import numpy as np
from scipy.sparse import hstack
import pandas as pd
from surprise import Dataset, SVD, Reader
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
warnings.filterwarnings("ignore")
ratings = pd.read_csv("Ratings.csv")
books = pd.read_csv("Books.csv")


def shift_row_right(row, start, column):
    for i in range(len(column) - 1, start, -1):
        row[column[i]] = row[column[i - 1]]
    row[column[start]] = np.nan
    return row


invalid_rows = books[books["Year-Of-Publication"].str.match("[^0-9]", na=False)]
columns = books.columns.tolist()
start_index = columns.index("Book-Author")
books.loc[invalid_rows.index] = books.loc[invalid_rows.index].apply(shift_row_right, axis=1, start=start_index, column=columns)
books = books.iloc[:, :-3]
books = books[books["ISBN"].isin(ratings["ISBN"])]
rating_counts = ratings["User-ID"].value_counts()
rating_not_one = rating_counts[rating_counts != 1].index
ratings = ratings[ratings["User-ID"].isin(rating_not_one)]
user_with_max_zeros = (ratings[ratings["Book-Rating"] == 0]["User-ID"].value_counts().idxmax())
reader = Reader(rating_scale=(1, 10))
non_zero_ratings = ratings[ratings["Book-Rating"] != 0]
data = Dataset.load_from_df(non_zero_ratings[["User-ID", "ISBN", "Book-Rating"]], reader)
train = data.build_full_trainset()
svd = SVD()
svd.fit(train)

user_ratings = ratings[ratings["User-ID"] == user_with_max_zeros]
zero_rated_books = user_ratings[user_ratings["Book-Rating"] == 0]["ISBN"].tolist()
recommendations = []
for isbn in zero_rated_books:
    pr = svd.predict(user_with_max_zeros, isbn)
    if pr.est >= 8:
        recommendations.append((isbn, pr.est))

average_ratings = ratings.groupby("ISBN")["Book-Rating"].mean().reset_index()
average_ratings.rename(columns={"Book-Rating": "Average-Rating"}, inplace=True)

final_data = pd.merge(books, average_ratings, on="ISBN", how="left").dropna(subset=["Average-Rating"])
X = final_data[["Book-Title", "Book-Author", "Publisher", "Year-Of-Publication"]]
y = final_data["Average-Rating"]

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_title = vectorizer.fit_transform(X["Book-Title"])

label_encoder_author = LabelEncoder()
label_encoder_publisher = LabelEncoder()

X_author = label_encoder_author.fit_transform(X["Book-Author"]).reshape(-1, 1)
X_publisher = label_encoder_publisher.fit_transform(X["Publisher"]).reshape(-1, 1)

scaler_year = StandardScaler()
X_year = scaler_year.fit_transform(X[["Year-Of-Publication"]])

scaler_author = StandardScaler()
scaler_publisher = StandardScaler()
X_author_scaled = scaler_author.fit_transform(X_author)
X_publisher_scaled = scaler_publisher.fit_transform(X_publisher)

X_combined = hstack([X_title, X_author_scaled, X_publisher_scaled, X_year])

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1))

X_train, X_test, y_train, y_test = train_test_split(X_combined, y_scaled, test_size=0.1, random_state=42)

model = SGDRegressor(max_iter=1000, tol=1e-3)
model.fit(X_train, y_train.ravel())

recommended_isbn = [isbn for isbn, _ in recommendations]
filtered_books = final_data[final_data["ISBN"].isin(recommended_isbn)].drop_duplicates(subset=["ISBN"])

X_filtered_title = vectorizer.transform(filtered_books["Book-Title"])
X_filtered_author = label_encoder_author.transform(filtered_books["Book-Author"]).reshape(-1, 1)
X_filtered_publisher = label_encoder_publisher.transform(filtered_books["Publisher"]).reshape(-1, 1)
X_filtered_year = scaler_year.transform(filtered_books[["Year-Of-Publication"]])

X_filtered_author_scaled = scaler_author.transform(X_filtered_author)
X_filtered_publisher_scaled = scaler_publisher.transform(X_filtered_publisher)

X_filtered_combined = hstack([X_filtered_title, X_filtered_author_scaled, X_filtered_publisher_scaled, X_filtered_year])
lin_reg_predictions = model.predict(X_filtered_combined)
lin_reg_predictions_unscaled = scaler_y.inverse_transform(lin_reg_predictions.reshape(-1, 1))

filtered_books["LinReg-Pred"] = lin_reg_predictions_unscaled

filtered_books["SVD_Pred"] = [pred for isbn, pred in recommendations if isbn in filtered_books["ISBN"].tolist()]

final_recommendations = filtered_books.sort_values(by=["LinReg-Pred"], ascending=False)

print(final_recommendations[["Book-Title", "LinReg-Pred", "SVD_Pred"]])

with open("svd.pkl", "wb") as f:
    pickle.dump(svd, f)

with open("linreg.pkl", "wb") as f:
    pickle.dump(model, f)

                                              Book-Title  LinReg-Pred  \
24282  The Lion, the Witch and the Wardrobe (Full-Col...     3.336055   
8463                            A Swiftly Tilting Planet     3.238639   
915                   The Giver (21st Century Reference)     3.222667   
2139   Harry Potter and the Sorcerer's Stone (Harry P...     3.180722   
3454    Harry Potter and the Chamber of Secrets (Book 2)     3.151806   
8666                           Walden and Other Writings     3.120717   
1131    The Witching Hour (Lives of the Mayfair Witches)     3.067587   
21738                          Goodnight Moon Board Book     3.053194   
2522                          Interview with the Vampire     3.037519   
1383                             A Prayer for Owen Meany     3.013705   
1195                                       Jurassic Park     2.978191   
1530                                      The Green Mile     2.973991   
16190      Key of Valor (Roberts, Nora. Key Trilogy